In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.bounds = -999999.0,999999.0
cobra_config.solver = "cplex"

In [4]:
#Loading BiGG's universal model for gapfilling

universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [5]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Lplantarum.tcds.top4.gramPosN.cim8.xml"

In [6]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [7]:
#Fixing masses
model.metabolites.get_by_id("3hcmrs7eACP_c").formula = "C25H45N2O9PRS"
model.metabolites.get_by_id("3hddecACP_c").formula = "C23H43N2O9PRS"
model.metabolites.get_by_id("3hmrsACP_c").formula = "C25H47N2O9PRS"
model.metabolites.get_by_id("ACP_c").formula = "C11H21N2O7PRS"
model.metabolites.get_by_id("apoACP_c").formula = "HOR"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("db4p_c").formula = "C4H7O6P"
model.metabolites.get_by_id("ddcaACP_c").formula = "C23H43N2O8PRS"
model.metabolites.get_by_id("dmlz_c").formula = "C13H18N4O6"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("fmnRD_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("gly_cys_c").formula = "C5H10N2O3S"
model.metabolites.get_by_id("gly_leu_c").formula = "C8H16N2O3"
model.metabolites.get_by_id("gly_phe_c").formula = "C11H14N2O3"
model.metabolites.get_by_id("gly_tyr_c").formula = "C11H14N2O4"
model.metabolites.get_by_id("lipoate_c").formula = "C8H14O2S2"
model.metabolites.get_by_id("lipoate_e").formula = "C8H14O2S2"
model.metabolites.get_by_id("myrsACP_c").formula = "C25H47N2O8PRS"
model.metabolites.get_by_id("phe__D_c").formula = "C9H11NO2"
model.metabolites.get_by_id("R_3hcmrs7ecoa_c").formula = "C35H56N7O18P3S"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"
model.metabolites.get_by_id("salchs4_c").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_e").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_p").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4fe_c").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_e").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_p").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("t3c7mrseACP_c").formula = "C25H43N2O8PRS"
model.metabolites.get_by_id("tdeACP_c").formula = "C25H45N2O8PRS"
model.metabolites.get_by_id("tde2coa_c").formula = "C35H54N7O17P3S" 
model.metabolites.get_by_id("tddec2eACP_c").formula = "C23H41N2O8PRS"
model.metabolites.get_by_id("tmrs2eACP_c").formula = "C25H45N2O8PRS"

In [8]:
#Fixing charges

model.metabolites.get_by_id("2agpg120_c").charge = -1
model.metabolites.get_by_id("2agpg120_p").charge = -1
model.metabolites.get_by_id("2agpg180_c").charge = -1
model.metabolites.get_by_id("2agpg180_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("3hmrsACP_c").charge = -1
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("apoACP_c").charge = 0
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("db4p_c").charge = -2
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("ddcaACP_c").charge = -1
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("enter_c").charge = 0
model.metabolites.get_by_id("enter_e").charge = 0
model.metabolites.get_by_id("enter_p").charge = 0
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fe3dcit_e").charge = -3
model.metabolites.get_by_id("feenter_c").charge = 3
model.metabolites.get_by_id("feenter_e").charge = 3
model.metabolites.get_by_id("feenter_p").charge = 3
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_e").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("lipoate_c").charge = 0
model.metabolites.get_by_id("lipoate_e").charge = 0
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("man6pglyc_c").charge = -3
model.metabolites.get_by_id("man6pglyc_e").charge = -3
model.metabolites.get_by_id("myrsACP_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa141_c").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("peptido_BS_c").charge = -2
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("rml1p_c").charge = -2
model.metabolites.get_by_id("sbt6p_c").charge = -2
model.metabolites.get_by_id("scys__L_c").charge = -1
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("td2coa_c").charge = -4
model.metabolites.get_by_id("tddec2eACP_c").charge = -1
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("tmrs2eACP_c").charge = -1
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_e").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2

In [9]:
# Identifying and removing duplicated reactions
# md model with removed reactions
# rd removed list
# dt reactions in doubt

[md,rd, dt] = removeDuplicateRxn(model)

In [10]:
#Removing reactions in doubt after manual inspection
md.remove_reactions([md.reactions.get_by_id("EX_abt__L_e"),md.reactions.get_by_id("EX_isetac_e"),
                     md.reactions.get_by_id("EX_glcn__D_e"),md.reactions.get_by_id("EX_ethso3_e"),
                     md.reactions.get_by_id("EX_galctr__D_e"),md.reactions.get_by_id("EX_sulfac_e"),
                     md.reactions.get_by_id("EX_orn__L_e"),md.reactions.get_by_id("EX_metsox_S__L_e"),
                     md.reactions.get_by_id("AMPAH"),md.reactions.get_by_id("CBPS_1"),
                     md.reactions.get_by_id("CHORS_1"),md.reactions.get_by_id("FMNAT_1"),
                     md.reactions.get_by_id("GLCNt2ir"),md.reactions.get_by_id("IG3PS_1"),
                     md.reactions.get_by_id("METSOX1abc"),md.reactions.get_by_id("MI1PP_1"),
                     md.reactions.get_by_id("NADK_1"),md.reactions.get_by_id("PRAMPC_1"),
                     md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("RBFK_1"),
                     md.reactions.get_by_id("RBK2")])

In [11]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
flux_variability_analysis(md,loopless=True)

,minimum,maximum
12PPDRte,0.000000,0.000000e+00
13PPDH,0.000000,0.000000e+00
2AGPE120tipp,0.000000,-1.313005e-12
2AGPE141tipp,0.000000,-4.206080e-13
2AGPE160tipp,0.000000,-6.184830e-12
...,...,...
NNDPR,0.001088,1.087533e-03
QULNS,0.001088,1.087533e-03
SUCBZL,0.000048,4.774066e-05
SUCBZS,0.000048,4.774068e-05


In [12]:
#Performing gapfill with universal model from BiGG to try to reduce blocked reactions
#Usually, it does not do anything (and it takes some time to run)

gapfill(md, universal, exchange_reactions=True, demand_reactions=False, iterations=10)

[[], [], [], [], [], [], [], [], [], []]

In [13]:
#Running FVA again after gapfilling
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arg__L_e,EX_arg__L_e,0.1412,[0; 0.1412],6,0.38%
asn__L_e,EX_asn__L_e,0.1151,[0; 4.494],4,0.21%
asp__L_e,EX_asp__L_e,0.9314,[0; 4.385],4,1.68%
ca2_e,EX_ca2_e,0.002485,[0.002361; 0.002485],0,0.00%
cl_e,EX_cl_e,0.002485,[0.002361; 0.002485],0,0.00%
cobalt2_e,EX_cobalt2_e,4.774E-05,[4.535E-05; 4.753E-05],0,0.00%
cu2_e,EX_cu2_e,0.0003385,[0.0003216; 0.0003385],0,0.00%
cys__L_e,EX_cys__L_e,0.04586,[0.03165; 2.236],3,0.06%
fe2_e,EX_fe2_e,0.006933,[0.006587; 0.006933],0,0.00%
fol_e,EX_fol_e,0.0003194,[0; 0.0003194],19,0.00%


In [14]:
#Adding/Removing/Editing reactions manually to reduce blocked reactions

md.remove_reactions([md.reactions.get_by_id("13PPDH"),md.reactions.get_by_id("BDH"),
                    md.reactions.get_by_id("BTS"),md.reactions.get_by_id("BUTKr"),
                    md.reactions.get_by_id("PBUTT"),md.reactions.get_by_id("LALDO")]) #Not connected in the network
md.remove_metabolites([md.metabolites.get_by_id("13ppd_c"),md.metabolites.get_by_id("3hppnl_c"),
                       md.metabolites.get_by_id("acac_c"),md.metabolites.get_by_id("bhb_c"),
                       md.metabolites.get_by_id("btal_c"),md.metabolites.get_by_id("btcoa_c"),
                       md.metabolites.get_by_id("btoh_c"),md.metabolites.get_by_id("but_c"),
                       md.metabolites.get_by_id("butpi_c"),md.metabolites.get_by_id("lald__D_c"),
                       md.metabolites.get_by_id("lgt__S_c")])

md.remove_reactions([md.reactions.get_by_id("3A2OA"),md.reactions.get_by_id("LASP2OA"),
                    md.reactions.get_by_id("YUMPS")]) #Dead-end. Nothing is done with msa_c, Largn_c
md.remove_metabolites([md.metabolites.get_by_id("msa_c"),md.metabolites.get_by_id("Largn_c"),
                      md.metabolites.get_by_id("psd5p_c")])

md.remove_reactions([md.reactions.get_by_id("GLUABUTt7pp")]) #Nothing is done with the metabolites in the periplasm
md.remove_metabolites([md.metabolites.get_by_id("glu__L_p"),md.metabolites.get_by_id("4abut_p")])

md.remove_reactions([md.reactions.get_by_id("St"),md.reactions.get_by_id("EX_s_e")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("s_e"),md.metabolites.get_by_id("s_c")])

md.remove_reactions([md.reactions.get_by_id("UREAt"),md.reactions.get_by_id("EX_urea_e")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("urea_e"),md.metabolites.get_by_id("urea_c")])

md.remove_reactions([md.reactions.get_by_id("EX_abt_e"),md.reactions.get_by_id("ARABRr")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("abt_e"),md.metabolites.get_by_id("abt_c")])

md.reactions.get_by_id("Growth").add_metabolites({md.metabolites.get_by_id("btn_c"): -2e-06})

md.remove_reactions([md.reactions.get_by_id("EX_galct__D_e")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("galct__D_e")])

md.remove_reactions([md.reactions.get_by_id("EX_glyb_e"),md.reactions.get_by_id("GLYBtex"),
                    md.reactions.get_by_id("GLYBabc"),md.reactions.get_by_id("GLYBt3pp")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("glyb_e"),md.metabolites.get_by_id("glyb_c"),
                      md.metabolites.get_by_id("glyb_p")])

md.remove_reactions([md.reactions.get_by_id("EX_ptrc_e"),md.reactions.get_by_id("PTRCabcpp"),
                    md.reactions.get_by_id("PTRCabc"),md.reactions.get_by_id("PTRCORNt7"),
                    md.reactions.get_by_id("PTRCtex")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("ptrc_e"),md.metabolites.get_by_id("ptrc_c"),
                      md.metabolites.get_by_id("ptrc_p")])

md.remove_reactions([md.reactions.get_by_id("SPMDabc"),md.reactions.get_by_id("SPMDtex"),
                    md.reactions.get_by_id("EX_spmd_e"),md.reactions.get_by_id("SPMDt3i"),
                    md.reactions.get_by_id("SPMDabcpp")]) #Nothing is done with the metabolites in the cytoplasm
md.remove_metabolites([md.metabolites.get_by_id("spmd_e"),md.metabolites.get_by_id("spmd_p"),
                      md.metabolites.get_by_id("spmd_c")])        

md.repair()

In [15]:
#Running FVA again after gapfilling and "pruning"
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arg__L_e,EX_arg__L_e,0.1412,[0; 0.1412],6,0.38%
asn__L_e,EX_asn__L_e,0.1151,[0; 4.494],4,0.21%
asp__L_e,EX_asp__L_e,0.9314,[0; 4.385],4,1.68%
btn_e,EX_btn_e,9.548E-07,[9.071E-07; 9.13E-07],10,0.00%
ca2_e,EX_ca2_e,0.002485,[0.002361; 0.002485],0,0.00%
cl_e,EX_cl_e,0.002485,[0.002361; 0.002485],0,0.00%
cobalt2_e,EX_cobalt2_e,4.774E-05,[4.535E-05; 4.713E-05],0,0.00%
cu2_e,EX_cu2_e,0.0003385,[0.0003216; 0.0003385],0,0.00%
cys__L_e,EX_cys__L_e,0.04586,[0.03165; 2.236],3,0.06%
fe2_e,EX_fe2_e,0.006933,[0.006587; 0.006933],0,0.00%


In [16]:
rlist = [md.reactions.get_by_id("6PHBG2"),md.reactions.get_by_id("ADK1"),md.reactions.get_by_id("ADK2"),
         md.reactions.get_by_id("AHSERL2"),md.reactions.get_by_id("ALCD19"),md.reactions.get_by_id("ALCD19y"),
         md.reactions.get_by_id("ARAT"),md.reactions.get_by_id("ASPT"),md.reactions.get_by_id("ASPTA"),
         md.reactions.get_by_id("CO2t"),md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),
         md.reactions.get_by_id("DR1ORx"),md.reactions.get_by_id("DR1ORy"),md.reactions.get_by_id("FUM"),
         md.reactions.get_by_id("G3PD1ir"),md.reactions.get_by_id("G3PD2"),md.reactions.get_by_id("G6PI"),
         md.reactions.get_by_id("G6PI3"),md.reactions.get_by_id("GALM1"),md.reactions.get_by_id("GK1"),
         md.reactions.get_by_id("GK2"),md.reactions.get_by_id("GLBRAN2"),md.reactions.get_by_id("GLBRAN3"),
         md.reactions.get_by_id("GLDBRAN2"),md.reactions.get_by_id("GLDBRAN3"),md.reactions.get_by_id("GLUDy"),
         md.reactions.get_by_id("GalMr"),md.reactions.get_by_id("GalMr_2"),md.reactions.get_by_id("H2CO3D"),
         md.reactions.get_by_id("H2CO3D2"),md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),
         md.reactions.get_by_id("H2Otpp"),md.reactions.get_by_id("HCO3E"),md.reactions.get_by_id("ILETA"),
         md.reactions.get_by_id("LDH_D"),md.reactions.get_by_id("LDH_L"),md.reactions.get_by_id("LEUTA"),
         md.reactions.get_by_id("LacR"),md.reactions.get_by_id("MDH"),md.reactions.get_by_id("METB1"),
         md.reactions.get_by_id("NADDP"),md.reactions.get_by_id("NADDPp_1"),md.reactions.get_by_id("NADK"),
         md.reactions.get_by_id("NADK1"),md.reactions.get_by_id("NADK2"),md.reactions.get_by_id("NADK3"),
         md.reactions.get_by_id("NADK4"),md.reactions.get_by_id("NADKd"),md.reactions.get_by_id("NDPK1"),
         md.reactions.get_by_id("NDPK4"),md.reactions.get_by_id("NDPK5"),md.reactions.get_by_id("NDPK7"),
         md.reactions.get_by_id("NDPK8"),md.reactions.get_by_id("NH4t"),md.reactions.get_by_id("NH4tex"),
         md.reactions.get_by_id("NH4tpp"),md.reactions.get_by_id("PGI"),md.reactions.get_by_id("PHETA1"),
         md.reactions.get_by_id("PPK2"),md.reactions.get_by_id("S6PG"),md.reactions.get_by_id("SHK3Dr"),
         md.reactions.get_by_id("SHSL1"),md.reactions.get_by_id("SHSL2"),md.reactions.get_by_id("SHSL2r"),
         md.reactions.get_by_id("SKDH_1"),md.reactions.get_by_id("TRPTA"),md.reactions.get_by_id("TYRTA"),
         md.reactions.get_by_id("UNK5"),md.reactions.get_by_id("araphe1"),md.reactions.get_by_id("araphe2"),
         md.reactions.get_by_id("araphe3"),md.reactions.get_by_id("aratry1"),md.reactions.get_by_id("aratry2"),
         md.reactions.get_by_id("aratyr1"),md.reactions.get_by_id("aratyr2"),md.reactions.get_by_id("aratyr3"),
         md.reactions.get_by_id("aratyr4")]

flux_variability_analysis(md,rlist,loopless=True)

,minimum,maximum
6PHBG2,0.000000e+00,0.000000
ADK1,0.000000e+00,0.688076
ADK2,-2.504574e-11,0.688076
AHSERL2,0.000000e+00,0.758954
ALCD19,-1.000000e+03,0.000000
...,...,...
aratry2,0.000000e+00,0.000000
aratyr1,0.000000e+00,0.000000
aratyr2,0.000000e+00,0.000000
aratyr3,0.000000e+00,0.000000


In [17]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Lplantarum.tcds.top4.gramPosN.cim8.'

In [18]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Lplantarum.tcds.top4.gramPosN.cim8.manual.xml
